# Market Basket Analysis

Mine interpretable product affinities and cross-sell rules from transaction baskets.

**Portfolio category:** Pattern mining

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from itertools import combinations

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Generate transparent transaction baskets

In [ ]:
bundles = [
    ["coffee", "milk", "sugar"],
    ["bread", "butter", "jam"],
    ["pasta", "tomato_sauce", "cheese"],
    ["diapers", "wipes", "baby_lotion"],
    ["chips", "salsa", "soft_drink"],
]
catalogue = sorted(set(item for bundle in bundles for item in bundle) | {"eggs", "tea", "rice", "soap"})
transactions = []
for _ in range(650):
    basket = set(bundles[rng.integers(0, len(bundles))])
    basket = {item for item in basket if rng.random() > 0.12}
    if rng.random() < 0.45:
        basket.add(rng.choice(catalogue))
    transactions.append(sorted(basket))
display(pd.DataFrame({"transaction": transactions}).head())

## 3. Basket quality and size

In [ ]:
basket_sizes = pd.Series([len(basket) for basket in transactions])
display(basket_sizes.describe().to_frame("basket_size"))
sns.histplot(basket_sizes, discrete=True)
plt.title("Basket-size distribution")
plt.tight_layout()

## 4. Frequent item support

In [ ]:
n_transactions = len(transactions)
item_counts = Counter(item for basket in transactions for item in set(basket))
item_support = pd.Series({item: count / n_transactions for item, count in item_counts.items()}).sort_values(ascending=False)
display(item_support.head(15).to_frame("support"))

## 5. Pair support and association rules

In [ ]:
pair_counts = Counter(
    pair
    for basket in transactions
    for pair in combinations(sorted(set(basket)), 2)
)
rules = []
for (left, right), count in pair_counts.items():
    pair_support = count / n_transactions
    for antecedent, consequent in [(left, right), (right, left)]:
        confidence = pair_support / item_support[antecedent]
        lift = confidence / item_support[consequent]
        rules.append({
            "antecedent": antecedent,
            "consequent": consequent,
            "support": pair_support,
            "confidence": confidence,
            "lift": lift,
        })
rules = pd.DataFrame(rules)
useful_rules = rules.query("support >= 0.04 and confidence >= 0.25 and lift > 1.05").sort_values(
    ["lift", "confidence"], ascending=False
)
display(useful_rules.head(20).round(3))

## 6. Rule quality checks

In [ ]:
coverage = useful_rules["antecedent"].nunique() / len(catalogue)
display(pd.Series({
    "candidate_rules": len(rules),
    "useful_rules": len(useful_rules),
    "antecedent_catalogue_coverage": coverage,
    "median_useful_lift": useful_rules["lift"].median(),
}).to_frame("value"))

## 7. Visualise the strongest rules

In [ ]:
plot_rules = useful_rules.head(15).copy()
plot_rules["rule"] = plot_rules["antecedent"] + " → " + plot_rules["consequent"]
sns.barplot(data=plot_rules, x="lift", y="rule", hue="confidence", palette="Blues", legend=False)
plt.title("Strongest association rules")
plt.tight_layout()

## 8. Business-ready rule table

In [ ]:
display(useful_rules.assign(
    recommendation=lambda frame: "Place " + frame["consequent"] + " near " + frame["antecedent"]
).head(12))

## 9. Key findings

Lift guards against recommending an already-popular item solely because it appears in many baskets.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For market basket analysis,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.